# Aula 1.5 — Validação cruzada (prática complementar)

**Disciplina:** Programação em Python para IA — IFNMG/Ceadi

Notebook **complementar** à videoaula: você avalia o modelo de forma **robusta** com **validação cruzada** (`cross_val_score`), em vez de confiar em um único split.

**No Colab:** faça upload de `triagem-covid-amostra-raw.csv` (arraste para a barra lateral). Rode as células na ordem.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

## 1. Carregar os dados e montar o Pipeline
Mesmo dataset e mesmo Pipeline do Módulo 1.4 (OneHot nas categóricas, StandardScaler na `idade`, sintomas direto).

In [2]:
df = pd.read_csv("triagem-covid-amostra-raw.csv")
y = df["target"]
X = df.drop(columns=["target"])

categoricas = ["estado", "racacor", "estacao"]
numericas = ["idade"]
pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("num", StandardScaler(), numericas),
    ],
    remainder="passthrough",
)
modelo = Pipeline([
    ("pre", pre),
    ("clf", MLPClassifier(hidden_layer_sizes=(128, 16), max_iter=300,
                           early_stopping=True, random_state=42)),
])

## 2. O problema: um único split
Com **uma** divisão treino/teste, você obtém **um** número. Ele depende da divisão sorteada.

In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
modelo.fit(X_tr, y_tr)
acc_split = accuracy_score(y_te, modelo.predict(X_te))
print("Acurácia (um único split):", round(acc_split, 3))

Acurácia (um único split): 0.594


## 3. Validação cruzada (k-fold)
`cross_val_score` divide os dados em **k partes** (folds). Cada parte é teste uma vez; o modelo é treinado e avaliado **k vezes**. Como passamos o **Pipeline**, o pré-processamento é ajustado **dentro de cada fold** (sem leakage).

In [4]:
scores = cross_val_score(modelo, X, y, cv=5, scoring="accuracy")
print("acurácia por fold:", scores.round(3))
print("média:", round(scores.mean(), 3))
print("desvio-padrão:", round(scores.std(), 3))

acurácia por fold: [0.632 0.601 0.586 0.64  0.6  ]
média: 0.612
desvio-padrão: 0.021


## 4. Comparar
Veja como o número de um único split é só um ponto dentro da faixa que a validação cruzada revela.

In [5]:
print("um único split:", round(acc_split, 3))
print(f"validação cruzada: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"faixa dos folds: {scores.min():.3f} a {scores.max():.3f}")

um único split: 0.594
validação cruzada: 0.612 ± 0.021
faixa dos folds: 0.586 a 0.640


## 5. Para pensar
Por que a média de 5 folds é uma estimativa mais confiável do que a acurácia de um único split? O que um **desvio-padrão alto** entre os folds indicaria sobre o modelo?